# 03. Stage 03: B2B Commercial Activation, Pitch Engine & Field Route Sequencing

**Executive B2B Outbound Intelligence Pipeline**

Converts resolved entity intelligence from Stage 02 into actionable commercial campaigns:
1. **Automation Guardrails & Domain Safety:** Classifies accounts into automated vs manual review sequences to protect email deliverability.
2. **Stratified B2B Pitch Generation:** Tailors cold email, LinkedIn ABM, and WhatsApp scripts by industry tier, workforce academic profile, and headcount.
3. **Field Visit Route Sequencing (TSP Optimization):** Solves nearest-neighbor + 2-opt TSP route schedules anchored at *The Coffee — Paseo MS*.
4. **Executive Diagnostic Tables:** Inline executive HTML tables (Tablas 1 a 4) for SDRs, field sales reps, and executive management.
5. **Clean-Room Dual Export:** Full private CRM/SDR artifacts + zero-PII public sector index.

In [1]:
# Cell 3.1: Ingest Resolved Entities, Actionable Sites & Data Contracts
import json
from pathlib import Path
import polars as pl
from IPython.display import HTML, display

def find_file(rel_path: str) -> Path:
    candidates = [Path(rel_path), Path(f"../{rel_path}"), Path(f"notebooks/{rel_path}")]
    for c in candidates:
        if c.exists(): return c
    return Path(rel_path)

# Ingest Stage 02 canonical account master and physical site action list
canon_path = find_file("data/processed/tables/stage2_canonical_action_master.parquet")
if not canon_path.exists():
    canon_path = find_file("data/processed/stage2_canonical_action_master.parquet")

site_path = find_file("data/processed/tables/stage2_site_action_list.parquet")
if not site_path.exists():
    site_path = find_file("data/processed/stage2_site_action_list.parquet")

df_canonical = pl.read_parquet(canon_path)
df_sites = pl.read_parquet(site_path)

# Ingest Data Contracts
chrono_path = find_file("data_contracts/scian_chronotype_mapping.json")
with open(chrono_path, "r", encoding="utf-8") as f:
    chronotype_mapping = json.load(f)

propensity_path = find_file("data_contracts/scian_b2b_propensity.json")
with open(propensity_path, "r", encoding="utf-8") as f:
    propensity_rules = json.load(f)

print(f" Loaded Stage-02 Canonical ABM Master: {len(df_canonical):,} unique corporate entities")
print(f" Loaded Stage-02 Physical Site Action List: {len(df_sites):,} operational locations")
print(f" Loaded Data Contracts: {len(chronotype_mapping)} chronotypes, {len(propensity_rules)} propensity weights")

 Loaded Stage-02 Canonical ABM Master: 108 unique corporate entities
 Loaded Stage-02 Physical Site Action List: 124 operational locations
 Loaded Data Contracts: 15 chronotypes, 15 propensity weights


In [2]:
# Cell 3.2: Operating Chronotypes, Domain Safety Guardrails & Tabla 1
# MIGRATES TO: src/mx_osint/activation/outreach_guardrails.py

def impute_operating_chronotype(scian_code: str | None) -> str:
    if not scian_code: return "Office_Standard_8AM"
    prefix2 = str(scian_code).strip()[:2]
    rule = chronotype_mapping.get(prefix2, {})
    return rule.get("chronotype", "Office_Standard_8AM")

def classify_outreach_guardrail(contact_risk: str | None, email_confidence: str | None) -> dict:
    risk = (contact_risk or "high").lower()
    conf = (email_confidence or "none").lower()
    if risk == "low" and conf in ["high", "medium-high"]:
        return {
            "guardrail_status": "automated_sequence_approved",
            "allowed_channel": "Automated Cold Email + LinkedIn",
            "risk_label": "BAJO (Reputación Protegida)",
            "risk_badge": "<span style='background:#dcfce7; color:#15803d; padding:2px 8px; border-radius:10px; font-weight:bold; font-size:10px;'>🟢 AUTOMATIZADO</span>",
            "recommendation": "Secuencia automática de 4 toques (Día 1, 3, 7, 14)."
        }
    elif risk == "medium":
        return {
            "guardrail_status": "manual_approval_required",
            "allowed_channel": "Campus Inherited / Semi-Automated",
            "risk_label": "MEDIO (Verificación Manual)",
            "risk_badge": "<span style='background:#fef3c7; color:#b45309; padding:2px 8px; border-radius:10px; font-weight:bold; font-size:10px;'>🟡 REVISIÓN MANUAL</span>",
            "recommendation": "Validar interlocutor en planta antes de disparo masivo."
        }
    else:
        return {
            "guardrail_status": "blocked_from_email_sequences",
            "allowed_channel": "LinkedIn Sales Nav + Phone Only",
            "risk_label": "ALTO (Riesgo de Rebote)",
            "risk_badge": "<span style='background:#fee2e2; color:#b91c1c; padding:2px 8px; border-radius:10px; font-weight:bold; font-size:10px;'>🔴 BLOQUEO EMAIL</span>",
            "recommendation": "Prohibido email directo. Canalizar por LinkedIn o llamada."
        }

guardrail_results = [
    classify_outreach_guardrail(r.get("contact_risk"), r.get("email_confidence"))
    for r in df_canonical.iter_rows(named=True)
]

df_canonical_guarded = df_canonical.with_columns([
    pl.Series("guardrail_status", [g["guardrail_status"] for g in guardrail_results]),
    pl.Series("allowed_outreach_channel", [g["allowed_channel"] for g in guardrail_results]),
    pl.Series("outreach_risk_label", [g["risk_label"] for g in guardrail_results]),
    pl.Series("outreach_risk_badge", [g["risk_badge"] for g in guardrail_results]),
    pl.Series("guardrail_action", [g["recommendation"] for g in guardrail_results]),
])

# Render Tabla 1: Resumen de Guardrails de Automatización
guard_counts = df_canonical_guarded.group_by(["outreach_risk_badge", "allowed_outreach_channel", "guardrail_action"]).agg(pl.len().alias("total_cuentas")).sort("total_cuentas", descending=True)

t1_rows = []
for r in guard_counts.iter_rows(named=True):
    pct = (r["total_cuentas"] / len(df_canonical_guarded)) * 100.0
    t1_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 8px 10px; text-align: center;">{r['outreach_risk_badge']}</td>
        <td style="padding: 8px 10px; font-weight: 600; color: #1e3a8a;">{r['allowed_outreach_channel']}</td>
        <td style="padding: 8px 10px; text-align: right; font-weight: bold; color: #0f766e;">{r['total_cuentas']} cuentas</td>
        <td style="padding: 8px 10px; text-align: right; color: #64748b;">{pct:.1f}%</td>
        <td style="padding: 8px 10px; font-size: 11px; color: #334155;">{r['guardrail_action']}</td>
    </tr>
    """)

html_table_1 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">🛡️ Tabla 1: Matriz de Guardrails de Automatización y Reputación de Dominio</h4>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 12px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #1e293b; color: white;">
            <tr>
                <th style="padding: 8px 10px; text-align: center;">Nivel de Riesgo</th>
                <th style="padding: 8px 10px;">Canal Autorizado</th>
                <th style="padding: 8px 10px; text-align: right;">Cuentas</th>
                <th style="padding: 8px 10px; text-align: right;">% del Total</th>
                <th style="padding: 8px 10px;">Regla Operativa para SDR / CRM</th>
            </tr>
        </thead>
        <tbody>
            {''.join(t1_rows)}
        </tbody>
    </table>
    <p style="font-size: 11px; color: #64748b; margin-top: 5px;"><i>Nota: Protege la entregabilidad del dominio evitando rebotes duros en inboxes no verificados.</i></p>
</div>
"""
display(HTML(html_table_1))

Nivel de Riesgo,Canal Autorizado,Cuentas,% del Total,Regla Operativa para SDR / CRM
🟢 AUTOMATIZADO,Automated Cold Email + LinkedIn,85 cuentas,78.7%,"Secuencia automática de 4 toques (Día 1, 3, 7, 14)."
🔴 BLOQUEO EMAIL,LinkedIn Sales Nav + Phone Only,17 cuentas,15.7%,Prohibido email directo. Canalizar por LinkedIn o llamada.
🟡 REVISIÓN MANUAL,Campus Inherited / Semi-Automated,6 cuentas,5.6%,Validar interlocutor en planta antes de disparo masivo.


In [3]:
# Cell 3.3: Stratified Multi-Channel B2B Pitch Generation & Tabla 2
# MIGRATES TO: src/mx_osint/activation/pitch_engine.py

def generate_account_pitch_pack(row: dict) -> dict:
    name = row.get("canonical_parent_name") or row.get("primary_site_name") or "la empresa"
    academic_tier = row.get("workforce_academic_level", "corporate_professional")
    headcount = int(row.get("total_est_employees_midpoint") or 25)
    sites = int(row.get("site_count") or 1)
    effective_em = row.get("effective_contact") or ""
    best_channel = row.get("best_contact_channel") or ""
    guardrail = row.get("guardrail_status", "")

    # Persona targeting
    if academic_tier in ["high_automation_engineering", "specialized_technical"]:
        dm_role = "Gerente de Planta / Abastecimiento Indirecto / RH Industrial"
        value_prop = "convenio de abastecimiento de café de especialidad y coffee breaks para juntas técnicas y cambios de turno"
        hook = f"Con más de {headcount:,} colaboradores en su complejo de Silao ({sites} sitio{'s' if sites > 1 else ''}), sabemos que la productividad en piso y la satisfacción del equipo de ingeniería son prioridades."
    elif academic_tier == "corporate_professional":
        dm_role = "Director de Finanzas / Compras Corporativas / Workplace Experience"
        value_prop = "barra ejecutiva de café de especialidad y servicio premium para salas de consejo y atención a clientes VIP"
        hook = f"Para optimizar la experiencia de los {headcount:,} profesionistas en sus oficinas corporativas de Silao."
    else:
        dm_role = "Administración de Oficina / Gerencia General"
        value_prop = "servicio ágil de café y coffee break continuo con entrega garantizada a pie de oficina"
        hook = f"Brindando servicio eficiente y constante para el flujo diario de sus {headcount:,} empleados."

    subject = f"[The Coffee Paseo MS] Propuesta de Café Corporativo & Coffee Breaks para {name}"
    
    email_body = (
        f"Estimado equipo de {name},\n\n"
        f"{hook}\n\n"
        f"Desde The Coffee — Paseo MS (Silao), ofrecemos {value_prop}, "
        f"con entregas puntuales y estándar de calidad de origen.\n\n"
        f"¿Tendrían 10 minutos esta semana para agendar una sesión de degustación sin costo para su equipo directivo?\n\n"
        f"Atentamente,\n"
        f"Equipo de Soluciones Corporativas\n"
        f"The Coffee — Paseo MS Silao"
    )

    linkedin_note = (
        f"Hola! Me pongo en contacto desde The Coffee en Paseo MS Silao. "
        f"Buscamos coordinar convenios de coffee break y café de especialidad para el equipo de {name}. "
        f"¿Podríamos platicar unos minutos?"
    )

    whatsapp_msg = (
        f"Hola, buenas tardes. Me comunico de The Coffee en Paseo MS Silao. "
        f"Queremos enviar un kit de degustación de café corporativo y catálogo de eventos "
        f"para el área de compras/RH de {name}. ¿Con quién podríamos canalizarlo?"
    )

    return {
        "target_decision_maker": dm_role,
        "email_subject": subject,
        "email_body": email_body,
        "linkedin_note": linkedin_note,
        "whatsapp_message": whatsapp_msg,
    }

pitches = [generate_account_pitch_pack(r) for r in df_canonical_guarded.iter_rows(named=True)]

df_canonical_activated = df_canonical_guarded.with_columns([
    pl.Series("target_decision_maker", [p["target_decision_maker"] for p in pitches]),
    pl.Series("email_subject", [p["email_subject"] for p in pitches]),
    pl.Series("email_body", [p["email_body"] for p in pitches]),
    pl.Series("linkedin_pitch", [p["linkedin_note"] for p in pitches]),
    pl.Series("whatsapp_pitch", [p["whatsapp_message"] for p in pitches]),
])

# Render Tabla 2: Top 15 Cuentas ABM con Pitch Generado
top_abm = df_canonical_activated.head(15)

t2_rows = []
for rank, r in enumerate(top_abm.iter_rows(named=True), start=1):
    sites_tag = f"<span style='background:#e0e7ff; color:#3730a3; padding:1px 6px; border-radius:10px; font-size:10px; font-weight:bold;'>{r['site_count']}</span>" if r['site_count'] > 1 else "1"
    em_fmt = f"<code style='color:#15803d; font-size:10px;'>{r['effective_contact']}</code>" if r['effective_contact'] else "<span style='color:#94a3b8;'>-</span>"
    t2_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 6px 8px; font-weight: bold; text-align: center; color: #64748b;">{rank}</td>
        <td style="padding: 6px 10px; font-weight: 600; color: #1e3a8a;">{r['canonical_parent_name']}</td>
        <td style="padding: 6px 8px; text-align: center;">{sites_tag}</td>
        <td style="padding: 6px 8px; text-align: center;">{r['outreach_risk_badge']}</td>
        <td style="padding: 6px 8px; font-size: 11px; color:#475569;">{r['target_decision_maker']}</td>
        <td style="padding: 6px 8px;">{em_fmt}</td>
        <td style="padding: 6px 8px; font-size: 11px; color:#0f766e; max-width: 250px; overflow: hidden; text-overflow: ellipsis; white-space: nowrap;" title="{r['email_subject']}">{r['email_subject']}</td>
        <td style="padding: 6px 8px; text-align: center;"><a href="{r.get('linkedin_url', '#')}" target="_blank" style="color:#0a66c2; text-decoration:none; font-weight:bold; font-size:11px;">🔍 ABM &rarr;</a></td>
    </tr>
    """)

html_table_2 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">💼 Tabla 2: Campañas de Prospección ABM (Top 15 Cuentas Canónicas con Pitches Personalizados)</h4>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 11px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #0f766e; color: white;">
            <tr>
                <th style="padding: 8px 8px; text-align: center;">#</th>
                <th style="padding: 8px 10px;">Cuenta Corporativa Canónica</th>
                <th style="padding: 8px 8px; text-align: center;">Sitios</th>
                <th style="padding: 8px 8px; text-align: center;">Guardrail</th>
                <th style="padding: 8px 8px;">Interlocutor Clave</th>
                <th style="padding: 8px 8px;">Contacto Resuelto</th>
                <th style="padding: 8px 8px;">Asunto de Email Personalizado</th>
                <th style="padding: 8px 8px; text-align: center;">Sales Nav</th>
            </tr>
        </thead>
        <tbody>
            {''.join(t2_rows)}
        </tbody>
    </table>
    <p style="font-size: 11px; color: #64748b; margin-top: 5px;"><i>Nota: Contenido completo de email, LinkedIn y WhatsApp exportado en stage3_abm_outreach_campaigns.csv.</i></p>
</div>
"""
display(HTML(html_table_2))

In [4]:
# Cell 3.4: Field Visit Route Sequencing (Nearest-Neighbor + 2-Opt TSP) & Tabla 3
# MIGRATES TO: src/mx_osint/activation/field_router.py
import math

PASEO_MS_LAT, PASEO_MS_LON = 20.9538, -101.4161

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2.0) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2.0) ** 2
    return 2.0 * r * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

# Corredor assignment
def assign_commercial_corridor(lat: float, lon: float, street: str | None) -> str:
    s = (street or "").upper()
    if lat > 20.970 or "FIPASI" in s or lon < -101.435:
        return "Corredor Industrial FIPASI / Carretera 45 Norte"
    elif lat < 20.940 or "PUERTO INTERIOR" in s or "COLINAS" in s:
        return "Corredor Guanajuato Puerto Interior / Las Colinas"
    else:
        return "Zona Urbana & Comercial Silao Centro"

corridors = [
    assign_commercial_corridor(float(r["lat"]), float(r["lon"]), r.get("street_name"))
    for r in df_sites.iter_rows(named=True)
]
df_sites_corridors = df_sites.with_columns(pl.Series("commercial_corridor", corridors))

# TSP Nearest-Neighbor + 2-Opt per corridor
def solve_tsp_corridor_route(sites_subset: pl.DataFrame, start_lat: float, start_lon: float) -> pl.DataFrame:
    recs = sites_subset.to_dicts()
    n = len(recs)
    if n == 0: return sites_subset

    unvisited = list(range(n))
    curr_lat, curr_lon = start_lat, start_lon
    route = []

    while unvisited:
        best_idx = min(unvisited, key=lambda idx: haversine_distance(curr_lat, curr_lon, float(recs[idx]["lat"]), float(recs[idx]["lon"])))
        route.append(best_idx)
        curr_lat, curr_lon = float(recs[best_idx]["lat"]), float(recs[best_idx]["lon"])
        unvisited.remove(best_idx)

    # 2-Opt local search
    def route_dist(r_order):
        d = haversine_distance(start_lat, start_lon, float(recs[r_order[0]]["lat"]), float(recs[r_order[0]]["lon"]))
        for k in range(len(r_order) - 1):
            d += haversine_distance(float(recs[r_order[k]]["lat"]), float(recs[r_order[k]]["lon"]), float(recs[r_order[k+1]]["lat"]), float(recs[r_order[k+1]]["lon"]))
        return d

    improved = True
    while improved:
        improved = False
        for i in range(len(route) - 1):
            for j in range(i + 1, len(route)):
                new_r = route[:i] + route[i:j+1][::-1] + route[j+1:]
                if route_dist(new_r) < route_dist(route) - 1e-4:
                    route = new_r
                    improved = True

    ordered = []
    cum_km = 0.0
    c_lat, c_lon = start_lat, start_lon
    for seq_num, idx in enumerate(route, start=1):
        r = dict(recs[idx])
        d_lat, d_lon = float(r["lat"]), float(r["lon"])
        leg_km = haversine_distance(c_lat, c_lon, d_lat, d_lon)
        cum_km += leg_km
        
        scian = str(r.get("scian_code", ""))[:2]
        chrono = chronotype_mapping.get(scian, {}).get("chronotype", "Office_Standard_8AM")
        if chrono == "Early_Bird_6AM":
            window = "06:30 - 08:30 (Cambio Turno Planta)"
        elif chrono == "Office_Standard_8AM":
            window = "09:00 - 11:30 (Recepción / Compras)"
        else:
            window = "11:30 - 14:00 (Mediodía / Comedor)"

        r["visit_sequence"] = seq_num
        r["leg_distance_km"] = round(leg_km, 2)
        r["cumulative_distance_km"] = round(cum_km, 2)
        r["est_drive_min"] = round(leg_km * 2.5)
        r["recommended_visit_window"] = window
        ordered.append(r)
        c_lat, c_lon = d_lat, d_lon

    return pl.DataFrame(ordered)

# Solve routes across priority corridors
fipasi_sites = df_sites_corridors.filter(pl.col("commercial_corridor").str.contains("FIPASI"))
gpi_sites = df_sites_corridors.filter(pl.col("commercial_corridor").str.contains("Puerto Interior"))
centro_sites = df_sites_corridors.filter(pl.col("commercial_corridor").str.contains("Urbana"))

routed_fipasi = solve_tsp_corridor_route(fipasi_sites, PASEO_MS_LAT, PASEO_MS_LON)
routed_gpi = solve_tsp_corridor_route(gpi_sites, PASEO_MS_LAT, PASEO_MS_LON)
routed_centro = solve_tsp_corridor_route(centro_sites, PASEO_MS_LAT, PASEO_MS_LON)

df_field_routes = pl.concat([routed_fipasi, routed_gpi, routed_centro])

# Render Tabla 3: Itinerario de Campo Top 15 Sitios Prioritarios
top_field = df_field_routes.sort("b2b_5d_score", descending=True).head(15)

t3_rows = []
for rank, r in enumerate(top_field.iter_rows(named=True), start=1):
    role_badge = "<span style='background:#dbeafe; color:#1e40af; padding:1px 6px; border-radius:4px; font-weight:bold; font-size:10px;'>Planta Principal</span>" if r.get("role_in_campus") == "primary_site" else "<span style='background:#f1f5f9; color:#475569; padding:1px 6px; border-radius:4px; font-size:10px;'>Sucursal / Bodega</span>"
    t3_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 6px 8px; font-weight: bold; text-align: center; color: #64748b;">#{r['visit_sequence']}</td>
        <td style="padding: 6px 10px; font-weight: 600; color: #1e3a8a;">{r['business_name']}<br><span style="font-size:10px; color:#64748b;">Matriz: {r['canonical_parent_name']}</span></td>
        <td style="padding: 6px 8px; text-align: center;">{role_badge}</td>
        <td style="padding: 6px 8px; font-size: 11px; color:#0f766e; font-weight:bold;">{r['commercial_corridor'].split(' ')[1]}</td>
        <td style="padding: 6px 8px; text-align: center; font-weight: bold; color: #0284c7;">{float(r.get('b2b_5d_score') or 0.0):.2f} pts</td>
        <td style="padding: 6px 8px; font-size: 11px; color:#15803d; font-weight:600;">{r['recommended_visit_window']}</td>
        <td style="padding: 6px 8px; text-align: right; font-weight:bold;">{r['leg_distance_km']} km</td>
        <td style="padding: 6px 8px; text-align: right; color:#64748b;">{r['cumulative_distance_km']} km ({r['est_drive_min']} min)</td>
    </tr>
    """)

html_table_3 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">🗺️ Tabla 3: Itinerario de Rutas Comerciales de Campo (Secuencia Optimizada de Visitas Presenciales)</h4>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 11px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #1e1b4b; color: white;">
            <tr>
                <th style="padding: 8px 8px; text-align: center;">Parada</th>
                <th style="padding: 8px 10px;">Establecimiento Físico & Matriz</th>
                <th style="padding: 8px 8px; text-align: center;">Rol de Sitio</th>
                <th style="padding: 8px 8px;">Corredor Industrial</th>
                <th style="padding: 8px 8px; text-align: center;">Score 5D</th>
                <th style="padding: 8px 8px;">Ventana de Visita Recomendada</th>
                <th style="padding: 8px 8px; text-align: right;">Tramo</th>
                <th style="padding: 8px 8px; text-align: right;">Acumulado</th>
            </tr>
        </thead>
        <tbody>
            {''.join(t3_rows)}
        </tbody>
    </table>
    <p style="font-size: 11px; color: #64748b; margin-top: 5px;"><i>Nota: Secuencia calculada desde The Coffee Paseo MS con algoritmo TSP 2-Opt. Itinerario completo en stage3_field_visit_routes.csv.</i></p>
</div>
"""
display(HTML(html_table_3))

Parada,Establecimiento Físico & Matriz,Rol de Sitio,Corredor Industrial,Score 5D,Ventana de Visita Recomendada,Tramo,Acumulado
#10,BANORTE SILAOMatriz: BANCO MERCANTIL DEL NORTE INSTITUCION DE BANCA MULTIPLE GRUPO FINANCIERO BANORTE,Planta Principal,Urbana,8.26 pts,09:00 - 11:30 (Recepción / Compras),0.11 km,1.78 km (0 min)
#4,SERVICIO DE CALIBRACION CAPYMETMatriz: CAPYMET,Planta Principal,Urbana,8.22 pts,09:00 - 11:30 (Recepción / Compras),0.62 km,0.91 km (2 min)
#20,CONSORCIO INDUSTRIAL MEXICANO DE AUTOPARTESMatriz: CONSORCIO INDUSTRIAL MEXICANO DE AUTOPARTES,Planta Principal,Urbana,8.04 pts,06:30 - 08:30 (Cambio Turno Planta),0.06 km,4.09 km (0 min)
#39,BANCO COMPARTAMOS SILAOMatriz: BANCO COMPARTAMOS INSTITUCION DE BANCA MULTIPLE,Planta Principal,Urbana,7.98 pts,09:00 - 11:30 (Recepción / Compras),0.03 km,8.83 km (0 min)
#19,LEAR SILAOMatriz: CONSORCIO INDUSTRIAL MEXICANO DE AUTOPARTES,Sucursal / Bodega,Urbana,7.97 pts,06:30 - 08:30 (Cambio Turno Planta),0.32 km,4.03 km (1 min)
#11,CAC SILAOMatriz: RADIO MOVIL DIPSA,Planta Principal,Urbana,7.81 pts,09:00 - 11:30 (Recepción / Compras),0.0 km,1.78 km (0 min)
#13,MAHLE SILAOMatriz: MAHLE BEHR RIO BRAVO,Planta Principal,Urbana,7.81 pts,06:30 - 08:30 (Cambio Turno Planta),1.34 km,3.14 km (3 min)
#7,AUMOVIO MEXICO MATRIZMatriz: AUMOVIO MEXICO,Planta Principal,Guanajuato,7.79 pts,06:30 - 08:30 (Cambio Turno Planta),0.21 km,2.61 km (1 min)
#17,JOHNANMatriz: JOHNAN DE MEXICO,Planta Principal,Urbana,7.79 pts,06:30 - 08:30 (Cambio Turno Planta),0.08 km,3.68 km (0 min)
#83,GENERAL MOTORS DE MEXICOMatriz: GENERAL MOTORS DE MEXICO,Planta Principal,Urbana,7.76 pts,06:30 - 08:30 (Cambio Turno Planta),0.71 km,18.14 km (2 min)


In [5]:
# Cell 3.5: Executive KPIs (Tabla 4), Folium Route Map & Clean-Room Dual Export
# MIGRATES TO: src/mx_osint/reporting/dual_export.py
import folium
from folium.plugins import MiniMap

# 1. Executive Deployment KPIs (Tabla 4)
total_accounts = len(df_canonical_activated)
auto_accounts = len(df_canonical_activated.filter(pl.col("guardrail_status") == "automated_sequence_approved"))
manual_accounts = len(df_canonical_activated.filter(pl.col("guardrail_status") == "manual_approval_required"))
abm_phone_accounts = len(df_canonical_activated.filter(pl.col("guardrail_status") == "blocked_from_email_sequences"))

total_sites = len(df_field_routes)
total_headcount = df_field_routes["est_employees_midpoint"].sum()
total_payroll_mxn = df_field_routes["est_monthly_payroll_mxn"].sum()

html_table_4 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">📊 Tabla 4: Métricas Ejecutivas del Despliegue Comercial (KPIs Stage 03)</h4>
    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-bottom: 12px;">
        <div style="background:#f8fafc; border:1px solid #cbd5e1; border-radius:6px; padding:10px; text-align:center;">
            <div style="font-size:10px; color:#64748b; font-weight:bold; text-transform:uppercase;">Cuentas ABM Canónicas</div>
            <div style="font-size:22px; font-weight:bold; color:#0f766e; margin-top:2px;">{total_accounts:,}</div>
            <div style="font-size:10px; color:#15803d; margin-top:2px;"><b>{auto_accounts}</b> listas para secuencia</div>
        </div>
        <div style="background:#f8fafc; border:1px solid #cbd5e1; border-radius:6px; padding:10px; text-align:center;">
            <div style="font-size:10px; color:#64748b; font-weight:bold; text-transform:uppercase;">Sitios Físicos en Ruta</div>
            <div style="font-size:22px; font-weight:bold; color:#1d4ed8; margin-top:2px;">{total_sites:,}</div>
            <div style="font-size:10px; color:#64748b; margin-top:2px;">3 corredores industriales</div>
        </div>
        <div style="background:#f8fafc; border:1px solid #cbd5e1; border-radius:6px; padding:10px; text-align:center;">
            <div style="font-size:10px; color:#64748b; font-weight:bold; text-transform:uppercase;">Plantilla Total Impactada</div>
            <div style="font-size:22px; font-weight:bold; color:#7c3aed; margin-top:2px;">{total_headcount:,}</div>
            <div style="font-size:10px; color:#64748b; margin-top:2px;">Empleados en Silao</div>
        </div>
        <div style="background:#f8fafc; border:1px solid #cbd5e1; border-radius:6px; padding:10px; text-align:center;">
            <div style="font-size:10px; color:#64748b; font-weight:bold; text-transform:uppercase;">Masa Salarial Mensual</div>
            <div style="font-size:22px; font-weight:bold; color:#b45309; margin-top:2px;">${total_payroll_mxn/1e6:.1f}M MXN</div>
            <div style="font-size:10px; color:#64748b; margin-top:2px;">Poder de compra corporativo</div>
        </div>
    </div>
</div>
"""
display(HTML(html_table_4))

# 2. Build Interactive Folium Field Route Map
m_route = folium.Map(location=[PASEO_MS_LAT, PASEO_MS_LON], zoom_start=13, tiles=None)

folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="&copy; OpenStreetMap contributors",
    name="🗺️ OpenStreetMap",
    control=True,
).add_to(m_route)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap &copy; CARTO",
    name="🏙️ CARTO Positron",
    control=True,
).add_to(m_route)

folium.LayerControl(position="topright").add_to(m_route)

# Base Anchor: The Coffee Paseo MS
folium.Marker(
    location=[PASEO_MS_LAT, PASEO_MS_LON],
    popup="<b>📍 The Coffee — Paseo MS</b><br>Centro Operativo de Rutas Comerciales",
    tooltip="📍 Base Comercial: The Coffee Paseo MS",
    icon=folium.Icon(color="red", icon="coffee", prefix="fa"),
).add_to(m_route)

corridor_colors = {
    "Corredor Industrial FIPASI / Carretera 45 Norte": "darkblue",
    "Corredor Guanajuato Puerto Interior / Las Colinas": "darkgreen",
    "Zona Urbana & Comercial Silao Centro": "purple",
}

for r in df_field_routes.iter_rows(named=True):
    corr = r.get("commercial_corridor", "")
    col = corridor_colors.get(corr, "blue")
    pop_html = f"""
    <div style='font-family: Arial; font-size: 11px; min-width: 220px;'>
        <b style='color:#0f172a;'>#{r['visit_sequence']} {r['business_name']}</b><br>
        <span style='color:#64748b;'>Matriz: {r['canonical_parent_name']}</span><hr style='margin:4px 0;'>
        <b>Score 5D:</b> {float(r.get('b2b_5d_score') or 0.0):.2f} pts<br>
        <b>Corredor:</b> {corr}<br>
        <b>Horario Visita:</b> {r['recommended_visit_window']}<br>
        <b>Distancia Tramo:</b> {r['leg_distance_km']} km ({r['est_drive_min']} min)<br>
        <a href='{r.get("linkedin_search_url", "#")}' target='_blank' style='color:#0a66c2; font-weight:bold;'>LinkedIn Search &rarr;</a>
    </div>
    """
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=6 if r.get("role_in_campus") == "primary_site" else 4,
        color=col,
        fill=True,
        fill_color=col,
        fill_opacity=0.85,
        popup=folium.Popup(pop_html, max_width=300),
        tooltip=f"#{r['visit_sequence']} {r['business_name']} ({r['recommended_visit_window'].split(' ')[0]})",
    ).add_to(m_route)

MiniMap(toggle_display=True).add_to(m_route)

# 3. Clean-Room Dual Export
for base_dir in [Path("data"), Path("../data"), Path("notebooks/data")]:
    if base_dir.exists():
        priv_dir = base_dir / "private"
        proc_dir = base_dir / "processed"
        priv_dir.mkdir(parents=True, exist_ok=True)
        proc_dir.mkdir(parents=True, exist_ok=True)

        # Private SDR / CRM exports
        df_canonical_activated.write_csv(priv_dir / "stage3_abm_outreach_campaigns.csv")
        df_field_routes.write_csv(priv_dir / "stage3_field_visit_routes.csv")
        m_route.save(str(priv_dir / "stage3_field_routes_map.html"))

        # Public Clean-Room Aggregate Export (Zero PII)
        pub_agg = (
            df_canonical_activated
            .group_by("workforce_academic_level")
            .agg([
                pl.len().alias("total_corporate_accounts"),
                pl.col("site_count").sum().alias("total_sites"),
                pl.col("total_est_employees_midpoint").sum().alias("total_employees"),
                pl.col("consolidated_payroll_mxn").sum().alias("total_payroll_mxn"),
                pl.col("guardrail_status").filter(pl.col("guardrail_status") == "automated_sequence_approved").len().alias("automated_ready_accounts"),
            ])
        )
        pub_index = {
            "metadata": {
                "generator": "mexico-commercial-osint",
                "stage": "03_outreach_routing",
                "version": "1.0.0",
                "base_anchor": "The Coffee — Paseo MS (Silao, GTO)",
                "total_accounts": len(df_canonical_activated),
                "total_sites": len(df_field_routes),
            },
            "industry_academic_aggregates": pub_agg.to_dicts(),
        }
        with open(proc_dir / "public_sector_index.json", "w", encoding="utf-8") as f:
            json.dump(pub_index, f, indent=2, ensure_ascii=False)

print(f"💾 Private CRM Campaigns exported: stage3_abm_outreach_campaigns.csv ({len(df_canonical_activated):,} accounts)")
print(f"💾 Private Field Route Schedule exported: stage3_field_visit_routes.csv ({len(df_field_routes):,} stops)")
print(f"💾 Private Interactive Route Map saved: stage3_field_routes_map.html")
print(f"💾 Public Clean-Room Sector Index exported: public_sector_index.json (Zero PII)")

display(m_route)

💾 Private CRM Campaigns exported: stage3_abm_outreach_campaigns.csv (108 accounts)
💾 Private Field Route Schedule exported: stage3_field_visit_routes.csv (124 stops)
💾 Private Interactive Route Map saved: stage3_field_routes_map.html
💾 Public Clean-Room Sector Index exported: public_sector_index.json (Zero PII)
